In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os, glob, copy, time
import numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.metrics import f1_score

device = 'cuda' if torch.cuda.is_available() else 'cpu'
IN04 = os.path.dirname(glob.glob('/kaggle/input/**/feats_b0_crop_frame.npy', recursive=True)[0])
meta = pd.read_csv(f'{IN04}/metadata.csv')
classes = meta.drop_duplicates('label').sort_values('label')['class'].tolist(); C = len(classes)

X  = torch.from_numpy(np.load(f'{IN04}/feats_b0_crop_frame.npy')).to(device)
Xf = torch.from_numpy(np.load(f'{IN04}/feats_b0_crop_frame_flip.npy')).to(device)
y  = torch.tensor(meta['label'].values, device=device)

def split_idx(s):
    return torch.tensor(np.where(meta['split_match'] == s)[0], device=device)
tr, va, te = split_idx('train'), split_idx('val'), split_idx('test')
print('Device:', device, '| features:', tuple(X.shape), '| train/val/test:', len(tr), len(va), len(te))

In [ ]:
class SpatialAttention(nn.Module):
    """Their 'spatial attention' (squeeze-and-excitation): learns a 0-1 weight for each of
    the 1280 feature channels and multiplies the features by it."""
    def __init__(self, d):
        super().__init__()
        self.att = nn.Sequential(nn.Linear(d, d // 8), nn.ReLU(), nn.Linear(d // 8, d), nn.Sigmoid())
    def forward(self, x):                      # x: (B, T, 1280)
        return x * self.att(x)

class TemporalAttentionOurs(nn.Module):
    def __init__(self, d, h=128):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(d, h), nn.Tanh(), nn.Linear(h, 1))
    def forward(self, H):
        a = torch.softmax(self.score(H).squeeze(-1), dim=1)
        return (a.unsqueeze(-1) * H).sum(1), a

class TemporalAttentionPaper(nn.Module):
    """Exactly as in sota_ours.py: Linear(D, D/4) -> ReLU -> Linear(D/4, 1) -> softmax over frames."""
    def __init__(self, d):
        super().__init__()
        self.score = nn.Sequential(nn.Linear(d, d // 4), nn.ReLU(), nn.Linear(d // 4, 1))
    def forward(self, H):
        a = torch.softmax(self.score(H).squeeze(-1), dim=1)
        return (a.unsqueeze(-1) * H).sum(1), a

class OurModel(nn.Module):
    """Our Notebook 03 Bi-GRU + attention, optionally with their spatial attention (change B)."""
    def __init__(self, num_classes, spatial_attn=False, in_dim=1280, hidden=256):
        super().__init__()
        self.se = SpatialAttention(in_dim) if spatial_attn else nn.Identity()
        self.norm, self.in_drop = nn.LayerNorm(in_dim), nn.Dropout(0.2)
        self.rnn = nn.GRU(in_dim, hidden, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
        self.attn = TemporalAttentionOurs(2 * hidden)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(2 * hidden, 256), nn.ReLU(),
                                  nn.Dropout(0.3), nn.Linear(256, num_classes))
    def forward(self, x):
        x = self.in_drop(self.norm(self.se(x)))
        z, a = self.attn(self.rnn(x)[0])
        return self.head(z), a

class PaperModel(nn.Module):
    """Head of sota_ours.py on top of B0 features: spatial attn -> 2-layer Bi-GRU(256, dropout 0.2)
    -> paper temporal attention -> Dropout(0.3) -> 512 -> ReLU -> Dropout(0.2) -> classes."""
    def __init__(self, num_classes, in_dim=1280, hidden=256):
        super().__init__()
        self.se = SpatialAttention(in_dim)
        self.rnn = nn.GRU(in_dim, hidden, num_layers=2, batch_first=True, dropout=0.2, bidirectional=True)
        self.attn = TemporalAttentionPaper(2 * hidden)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(2 * hidden, 512), nn.ReLU(),
                                  nn.Dropout(0.2), nn.Linear(512, num_classes))
    def forward(self, x):
        z, a = self.attn(self.rnn(self.se(x))[0])
        return self.head(z), a

for f in [lambda: OurModel(C), lambda: OurModel(C, True), lambda: PaperModel(C)]:
    m = f().to(device)
    print(type(m).__name__, f'{sum(p.numel() for p in m.parameters()) / 1e6:.2f}M params',
          tuple(m(torch.randn(2, 16, 1280, device=device))[0].shape))

In [ ]:
@torch.no_grad()
def evaluate(model, ids, crit=None, bs=512):
    model.eval()
    logits = torch.cat([model(X[ids[i:i+bs]].float())[0] for i in range(0, len(ids), bs)])
    yt = y[ids]; pred = logits.argmax(1)
    return {'loss': crit(logits, yt).item() if crit else None,
            'acc': (pred == yt).float().mean().item(),
            'top3': (logits.topk(3, 1).indices == yt[:, None]).any(1).float().mean().item(),
            'macro_f1': f1_score(yt.cpu(), pred.cpu(), average='macro')}

def train(make_model, seed, bs=64, epochs=40, lr=1e-3, sched='plateau', select='f1', patience=8):
    """sched='plateau', select='f1'  -> our Notebook 03 recipe
       sched='step',    select='paper'-> their recipe: StepLR(10, 0.5), early stop on val loss, keep best val acc"""
    torch.manual_seed(seed); np.random.seed(seed)
    counts = torch.bincount(y[tr], minlength=C).float()
    crit = nn.CrossEntropyLoss(weight=counts.sum() / (C * counts), label_smoothing=0.1)
    model = make_model().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = (torch.optim.lr_scheduler.StepLR(opt, step_size=10, gamma=0.5) if sched == 'step'
                 else torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode='max', factor=0.5, patience=3))
    best_score, best_vloss, best_state, bad = -1, float('inf'), None, 0
    for ep in range(1, epochs + 1):
        model.train()
        perm = tr[torch.randperm(len(tr), device=device)]
        for i in range(0, len(perm), bs):
            b = perm[i:i+bs]; xb = X[b].float()
            m = torch.rand(len(b), device=device) < 0.5          # clip-level flip augmentation
            xb[m] = Xf[b[m]].float()
            loss = crit(model(xb)[0], y[b])
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        val = evaluate(model, va, crit)
        if sched == 'step': scheduler.step()
        else: scheduler.step(val['macro_f1'])
        if select == 'f1':                                       # ours: select + stop on val macro-F1
            if val['macro_f1'] > best_score:
                best_score, best_state, bad = val['macro_f1'], copy.deepcopy(model.state_dict()), 0
            else:
                bad += 1
        else:                                                    # theirs: keep best val acc, stop on val loss
            if val['acc'] > best_score:
                best_score, best_state = val['acc'], copy.deepcopy(model.state_dict())
            if val['loss'] < best_vloss: best_vloss, bad = val['loss'], 0
            else: bad += 1
        if bad >= patience:
            break
    model.load_state_dict(best_state)
    return model, evaluate(model, va, crit), evaluate(model, te, crit), ep

In [ ]:
EXPS = {
    'Ours: Bi-GRU + attn':                  (lambda: OurModel(C),                    {}),
    'Ours + spatial attn (B)':              (lambda: OurModel(C, spatial_attn=True), {}),
    'Paper head + paper training (B + C)':  (lambda: PaperModel(C),
                                             dict(bs=32, epochs=30, sched='step', select='paper', patience=10)),
}
rows, saved = [], {}
for name, (make, kw) in EXPS.items():
    for s in [0, 1, 2]:
        t0 = time.time()
        model, val, test, ep = train(make, s, **kw)
        rows.append({'model': name, 'seed': s, 'epochs': ep, 'val_acc': val['acc'], 'val_f1': val['macro_f1'],
                     'test_acc': test['acc'], 'test_top3': test['top3'], 'test_f1': test['macro_f1']})
        if s == 0: saved[name] = model
        print(f"{name:38s} seed {s}: test acc {test['acc']:.3f} f1 {test['macro_f1']:.3f} "
              f"({ep} ep, {time.time() - t0:.0f}s)")

res = pd.DataFrame(rows)
res.to_csv('/kaggle/working/results_stage1_runs.csv', index=False)
summary = res.groupby('model', sort=False).agg(
    val_f1=('val_f1', 'mean'), test_acc=('test_acc', 'mean'), acc_std=('test_acc', 'std'),
    test_top3=('test_top3', 'mean'), test_f1=('test_f1', 'mean')).round(4)
summary.to_csv('/kaggle/working/results_stage1_summary.csv')
summary

In [ ]:
torch.save({'state_dict': saved['Paper head + paper training (B + C)'].state_dict(), 'classes': classes},
           '/kaggle/working/paper_head_frozen.pt')
torch.save({'state_dict': saved['Ours + spatial attn (B)'].state_dict(), 'classes': classes},
           '/kaggle/working/ours_se_frozen.pt')
!ls -lh /kaggle/working